# 확장 EDA v1 — 2026-05-21

`shared/eda.ipynb` (1차 EDA) 의 후속 분석. 결과 요약은 [`docs/eda_findings.md §15`](../docs/eda_findings.md#15-확장-eda--미해결-질문-분석-2026-05-21) 참조.

## 핵심 발견

### 강력 시그널 (★★★) — Two-stage reranker (1차 추천 → 2차 reranking 모델) 의 우선 입력 feature 후보

1. **가격대 일치** — purchase 의 **86.8%** 가 user 가 평소 view 한 가격 범위 안 (§2.1)
2. **카테고리 선호도** — user 가 가장 많이 view 한 카테고리 ↔ 가장 많이 purchase 한 카테고리 일치율 **74.8%** (우연 (random baseline) 50%) (§2.3)
3. **브랜드 선호도** — user 가 가장 많이 view 한 브랜드 ↔ 가장 많이 purchase 한 브랜드 일치율 **46.5%** (우연 (random baseline) 33%) (§2.2)
4. **사전 view 이력의 강력함** — spike 기간 구매자의 **90.5%** 가 spike 이전 view 이력 보유 → 신규 user 거의 없음, 순차 모델 (SASRec 계열) 적합 (§1.1)
5. **서로 다른 session 의 view → purchase** — view → purchase 의 **94.6%** 가 같은 session 안이 아닌 다른 session 에서 일어남 → user 의 장기 행동 기록 전체를 보는 모델 우선 (§4.1)

### 중간 시그널 (★★)

6. **item 별 전환율 (보정 필수)** — user 별 view→purchase 비율은 **99.5%** 가 0 (구매가 너무 적어 신호 거의 없음). item 별 (view≥100, n=12.7k) 통계 보정 (Bayesian smoothing) 거친 값만 신뢰 (§3.2)

### 약한 시그널 / 노이즈 (☆ / ⚠️)

7. **요일 / 시간대 효과** — spike (Feb 27-29) 빼면 거의 없음 → 주기성 feature (sin/cos 변환) 추가 가치 낮음 (§3.1)
8. **브랜드 × 카테고리 매핑 anomaly** — IT 브랜드가 **100% apparel** 로 매핑, 전체 브랜드의 **71.1%** 가 단일 카테고리 — 두 변수 조합 feature 는 새 정보 없음 (§1.2)

## 데이터 anomaly 메모

§1.2 에서 발견:
- IT 브랜드 (xiaomi/sony/iqos/samsung/apple) 가 view/cart/purchase 전 event type 에서 **100% apparel** 카테고리
- 전체 1,859 brand 중 **71.1% (1,321 brand) 가 단일 category(l2) 에만 매핑**
- brand 와 category 가 거의 1:1 매핑 — 모델에서 cross-feature 새 정보 거의 없음
- ⚠️ **라벨의 실세계 의미는 안 맞지만, 데이터 안에서는 일관되게 쓰임** (xiaomi 라벨이 view → cart → purchase 모두에서 같은 의미). brand_id / category_id 자체를 학습하는 모델 (SASRec/ALS/EASE/reranker) 은 안전 — Tier 1 시그널 유효. 단 외부 의미 정보 (word2vec / Wikidata 같은 외부 지식) 끌어오는 시도는 금지

**모델링 영향**:
- ID embedding 기반 (SASRec / ALS / EASE): 영향 없음
- Brand text embedding (word2vec init 등): 사용 금지 — 의미 일치 안 함
- Two-stage reranker 의 brand × category cross feature: 새 정보 없으므로 후순위

## 다음 액션

1. **Two-stage reranker 입력 feature 정렬** — 강력 시그널 3개 (가격대 + 카테고리 선호도 + 브랜드 선호도) 우선 입력
2. **순차 모델 (SASRec / TiSASRec / MB-STR) 선택** — 서로 다른 session 의 패턴 + 사전 view 이력 모두 정합
3. **session 단위 모델 (SR-GNN 등) 우선순위 ↓** — view ↔ purchase 가 다른 session 에서 일어남
4. **주기성 feature / 브랜드 이름 임베딩 시도 X** — 효과 약함 / anomaly 영향

## 분석 항목

| Section | 주제 | 강도 |
|---|---|:---:|
| 1.1 | Feb 27-29 spike 정체 (카테고리 / 브랜드 / 구매자) | — |
| 1.2 | 브랜드 × 카테고리 매핑 anomaly | ⚠️ |
| 2.1 | 가격대 일치 (price band) | ★★★ |
| 2.2 | 브랜드 선호도 (brand affinity) | ★★★ |
| 2.3 | 카테고리 선호도 (category affinity) | ★★★ |
| 3.1 | 시간대 / 요일 | ☆ |
| 3.2 | 전환율 (item 별 보정 필요) | ★★ |
| 4.1 | 서로 다른 session 의 view → purchase 패턴 | ★★★ |

연관 문서: [`docs/eda_findings.md §15`](../docs/eda_findings.md) / [`docs/candidate_models.md §7`](../docs/candidate_models.md)

## 0. Setup — load data + derive time features

In [25]:
from pathlib import Path
import numpy as np
import pandas as pd

# 서버: /root/data/train.parquet, 로컬: baseline/data/train.parquet
DATA_PATH = Path('/root/data/train.parquet')
if not DATA_PATH.exists():
    DATA_PATH = Path('../baseline/data/train.parquet')
if not DATA_PATH.exists():
    DATA_PATH = Path('baseline/data/train.parquet')

df = pd.read_parquet(DATA_PATH)
df['event_time'] = pd.to_datetime(df['event_time'], format='%Y-%m-%d %H:%M:%S %Z', utc=True)
df['date'] = df['event_time'].dt.date
df['dow']  = df['event_time'].dt.dayofweek  # 0=Mon
df['hour'] = df['event_time'].dt.hour

spike_mask = df['date'].between(pd.Timestamp('2020-02-27').date(),
                                pd.Timestamp('2020-02-29').date())
base_mask  = df['date'].between(pd.Timestamp('2020-02-01').date(),
                                pd.Timestamp('2020-02-26').date())

print(f'shape: {df.shape}')
print(f'event_time range: {df["event_time"].min()} .. {df["event_time"].max()}')
print(df['event_type'].value_counts())


shape: (8350311, 11)
event_time range: 2019-11-01 00:00:17+00:00 .. 2020-02-29 23:59:33+00:00
event_type
view        8331873
cart          16362
purchase       2076
Name: count, dtype: int64


## 1. 데이터 특성 진단

### 1.1 Feb 27-29 spike 정체

가설: spike (purchase 1,437건, 전체 69%) 가 특정 카테고리/브랜드에 집중? 구매자는 기존 user vs 신규?

In [26]:
purchases_spike = df[spike_mask & (df['event_type'] == 'purchase')]
purchases_base  = df[base_mask  & (df['event_type'] == 'purchase')]
print(f'spike purchases: {len(purchases_spike)}, base (Feb 1-26) purchases: {len(purchases_base)}')

cat_l2 = lambda s: s.fillna('(none)').str.split('.').str[:2].str.join('.')
print('\nspike category_code (l2) top 15:')
print(cat_l2(purchases_spike['category_code']).value_counts().head(15))
print('\nbase  category_code (l2) top 15:')
print(cat_l2(purchases_base['category_code']).value_counts().head(15))

print('\nspike brand top 15:')
print(purchases_spike['brand'].fillna('(none)').value_counts().head(15))
print('\nbase  brand top 15:')
print(purchases_base['brand'].fillna('(none)').value_counts().head(15))

# 구매자 분포 — 사전 이력 보유 여부
spike_users = set(purchases_spike['user_id'].unique())
pre_spike = df[base_mask | df['date'].between(pd.Timestamp('2019-11-01').date(),
                                              pd.Timestamp('2020-01-31').date())]
pre_view_users = set(pre_spike[pre_spike['event_type'] == 'view']['user_id'].unique())
pre_any_users  = set(pre_spike['user_id'].unique())

n_view = len(spike_users & pre_view_users)
n_new  = len(spike_users - pre_any_users)
print(f'\nspike unique 구매자: {len(spike_users)}')
print(f'  사전 view 이력 보유: {n_view} ({100*n_view/len(spike_users):.1f}%)')
print(f'  사전 이력 0 (신규):   {n_new} ({100*n_new/len(spike_users):.1f}%)')


spike purchases: 1437, base (Feb 1-26) purchases: 148

spike category_code (l2) top 15:
category_code
apparel.shoes        1010
apparel.trousers      136
apparel.shirt          63
apparel.costume        59
apparel.scarf          40
apparel.underwear      34
apparel.tshirt         32
apparel.shorts         30
apparel.jeans          15
apparel.sock            7
apparel.glove           5
apparel.pajamas         2
apparel.skirt           2
apparel.jacket          1
apparel.jumper          1
Name: count, dtype: int64

base  category_code (l2) top 15:
category_code
apparel.shoes        106
apparel.trousers       9
apparel.costume        8
apparel.shorts         8
apparel.scarf          6
apparel.underwear      4
apparel.tshirt         3
apparel.shirt          2
apparel.skirt          1
apparel.dress          1
Name: count, dtype: int64

spike brand top 15:
brand
xiaomi       171
sony         122
iqos          96
samsung       81
apple         45
defacto       45
luminarc      34
glo         

### 1.2 brand × category 매핑 anomaly

§1.1 에서 발견: spike top brand 가 xiaomi/sony 등 IT 인데 카테고리는 모두 apparel. 전체 데이터에서 일관적인가? brand 의 매핑 다양성은?

In [27]:
# (a) IT brand 의 event_type 별 카테고리 일관성
it_brands = ['xiaomi', 'sony', 'iqos', 'samsung', 'apple']
print('IT 브랜드들의 event_type 별 category_code (l1) 분포:')
for et in ['view', 'cart', 'purchase']:
    sub = df[(df['brand'].isin(it_brands)) & (df['event_type'] == et)]
    cat_l1 = sub['category_code'].fillna('(none)').str.split('.').str[0]
    top = cat_l1.value_counts().head(3)
    print(f'  [{et}] n={len(sub):,}: {dict(top)}')

# (b) 전체 brand 의 매핑 다양성
brand_df = df.dropna(subset=['brand', 'category_code']).copy()
brand_df['cat_l2'] = brand_df['category_code'].str.split('.').str[:2].str.join('.')
brand_cat_div = brand_df.groupby('brand')['cat_l2'].nunique()
print(f'\n전체 brand 수: {len(brand_cat_div):,}')
print(f'  단일 category(l2) 만 매핑되는 brand: {(brand_cat_div == 1).sum():,} ({100*(brand_cat_div == 1).sum()/len(brand_cat_div):.1f}%)')
print(f'  brand 당 unique category 분포: median={brand_cat_div.median():.0f}, p90={brand_cat_div.quantile(0.9):.0f}, max={brand_cat_div.max()}')


IT 브랜드들의 event_type 별 category_code (l1) 분포:
  [view] n=1,031,067: {'apparel': np.int64(1031067)}
  [cart] n=5,977: {'apparel': np.int64(5977)}
  [purchase] n=668: {'apparel': np.int64(668)}

전체 brand 수: 1,859
  단일 category(l2) 만 매핑되는 brand: 1,321 (71.1%)
  brand 당 unique category 분포: median=1, p90=3, max=14


## 2. 강력 시그널 (★★★ reranker Tier 1)

### 2.1 가격대 (price band)

가설: user 는 자기 평소 view price 대역 안에서 구매?

In [28]:
for et in ['view', 'cart', 'purchase']:
    sub = df.loc[df['event_type'] == et, 'price']
    print(f'[{et}] n={len(sub):>9,}  median={sub.median():>7.2f}  mean={sub.mean():>7.2f}  p90={sub.quantile(0.9):>7.2f}')

# user-level price band
user_view = df[df['event_type'] == 'view'].groupby('user_id')['price'].agg(
    n='size', p_med='median',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
)
user_view['iqr'] = user_view['p75'] - user_view['p25']
active = user_view[user_view['n'] >= 5]
user_purch = df[df['event_type'] == 'purchase'].groupby('user_id')['price'].agg(purch_med='median')
j = active.join(user_purch, how='inner')
print(f'\nview>=5 + has purchase user: {len(j):,}')
if len(j) > 0:
    within_iqr = j['purch_med'].between(j['p25'], j['p75']).sum()
    lo = j['p_med'] - 1.5 * j['iqr']
    hi = j['p_med'] + 1.5 * j['iqr']
    within_15 = j['purch_med'].between(lo, hi).sum()
    print(f'  purchase ∈ view IQR (p25~p75):     {within_iqr} ({100*within_iqr/len(j):.1f}%)')
    print(f'  purchase ∈ view median ± 1.5*IQR:  {within_15} ({100*within_15/len(j):.1f}%)  ← reranker feature 강력')


[view] n=8,331,873  median=  79.02  mean= 150.93  p90= 384.26
[cart] n=   16,362  median=  65.64  mean= 149.42  p90= 387.09
[purchase] n=    2,076  median=  64.24  mean= 122.29  p90= 383.90

view>=5 + has purchase user: 1,397
  purchase ∈ view IQR (p25~p75):     776 (55.5%)
  purchase ∈ view median ± 1.5*IQR:  1212 (86.8%)  ← reranker feature 강력


### 2.2 brand affinity

가설: view 가장 많이 한 brand 가 가장 많이 사는 brand?

In [29]:
purch_brand = df[df['event_type'] == 'purchase'].dropna(subset=['brand'])
purch_users = purch_brand['user_id'].unique()

user_view_top = (df[(df['event_type'] == 'view') & (df['user_id'].isin(purch_users))]
                 .dropna(subset=['brand'])
                 .groupby('user_id')['brand']
                 .agg(lambda x: x.value_counts().index[0])
                 .rename('view_top'))
user_purch_top = (purch_brand.groupby('user_id')['brand']
                  .agg(lambda x: x.value_counts().index[0])
                  .rename('purch_top'))
j = pd.concat([user_view_top, user_purch_top], axis=1).dropna()
match = (j['view_top'] == j['purch_top']).sum()
print(f'view top brand ↔ purchase top brand 일치율: {match}/{len(j)} = {100*match/len(j):.1f}%')

# random baseline
uniq = df.dropna(subset=['brand']).groupby('user_id')['brand'].nunique()
print(f'\nuser 별 unique brand 수: median={uniq.median():.0f}, p90={uniq.quantile(0.9):.0f}')
print(f'  random baseline (1/median): ≈ {100/uniq.median():.0f}%')


view top brand ↔ purchase top brand 일치율: 743/1597 = 46.5%

user 별 unique brand 수: median=3, p90=11
  random baseline (1/median): ≈ 33%


### 2.3 category affinity

§2.2 brand 분석을 category 로 확장.

In [ ]:
purch_df = df[df['event_type'] == 'purchase'].dropna(subset=['category_code']).copy()
purch_df['cat_l2'] = purch_df['category_code'].str.split('.').str[:2].str.join('.')
purch_users = purch_df['user_id'].unique()

user_view_top_cat = (df[(df['event_type'] == 'view') & (df['user_id'].isin(purch_users))]
                     .dropna(subset=['category_code'])
                     .assign(cat_l2=lambda d: d['category_code'].str.split('.').str[:2].str.join('.'))
                     .groupby('user_id')['cat_l2']
                     .agg(lambda x: x.value_counts().index[0])
                     .rename('view_top_cat'))
user_purch_top_cat = (purch_df.groupby('user_id')['cat_l2']
                      .agg(lambda x: x.value_counts().index[0])
                      .rename('purch_top_cat'))
j = pd.concat([user_view_top_cat, user_purch_top_cat], axis=1).dropna()
match = (j['view_top_cat'] == j['purch_top_cat']).sum()
print(f'view top category ↔ purchase top category 일치율: {match}/{len(j)} = {100*match/len(j):.1f}%')

# random baseline
user_cat_uniq = (df[df['user_id'].isin(purch_users)]
                 .dropna(subset=['category_code'])
                 .assign(cat_l2=lambda d: d['category_code'].str.split('.').str[:2].str.join('.'))
                 .groupby('user_id')['cat_l2'].nunique())
print(f'\nuser 별 unique category(l2) 수: median={user_cat_uniq.median():.0f}, mean={user_cat_uniq.mean():.1f}')
print(f'  random baseline (1/median): ≈ {100/user_cat_uniq.median():.0f}%')
cat_baseline = 100 / user_cat_uniq.median()
cat_pct = 100 * match / len(j)
print(f'\n비교:')
print(f'  brand    affinity 일치율 46.5% (random ~33%) → lift 1.4x')
print(f'  category affinity 일치율 {cat_pct:.1f}% (random ~{cat_baseline:.0f}%) → lift {cat_pct/cat_baseline:.1f}x')


## 3. 약한 시그널 (☆ / ⚠️)

### 3.1 시간대 / 요일

가설: 요일/시간대 효과가 있나? spike 영향 분리해서 봐야.

In [31]:
def event_table(d, by):
    t = d.groupby([by, 'event_type']).size().unstack(fill_value=0)
    t['total'] = t.sum(axis=1)
    if 'purchase' in t.columns:
        t['purchase_rate'] = t['purchase'] / t['total']
    return t

print('=== 요일별 (full data) ===')
t_full = event_table(df, 'dow')
t_full.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print(t_full)
print('\n→ Thu/Fri/Sat 가 비정상적으로 높음 (Feb 27=Thu, 28=Fri, 29=Sat 이 spike)')

print('\n=== 요일별 (spike Feb 27-29 제외) ===')
t_ns = event_table(df[~spike_mask], 'dow')
t_ns.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print(t_ns)
print('\n→ spike 빼면 요일 효과 거의 없음 → cyclic encoding 가치 낮음')

print('\n=== 시간대별 UTC (full data) ===')
print(event_table(df, 'hour'))


=== 요일별 (full data) ===
event_type  cart  purchase     view    total  purchase_rate
Mon         1463        67  1180667  1182197       0.000057
Tue         1340        60  1140053  1141453       0.000053
Wed         1194        80  1118286  1119560       0.000071
Thu         3200       792  1099650  1103642       0.000718
Fri         4443       505  1205529  1210477       0.000417
Sat         2862       443  1274008  1277313       0.000347
Sun         1860       129  1313680  1315669       0.000098

→ Thu/Fri/Sat 가 비정상적으로 높음 (Feb 27=Thu, 28=Fri, 29=Sat 이 spike)

=== 요일별 (spike Feb 27-29 제외) ===
event_type  cart  purchase     view    total  purchase_rate
Mon         1463        67  1180667  1182197       0.000057
Tue         1340        60  1140053  1141453       0.000053
Wed         1194        80  1118286  1119560       0.000071
Thu         3200       124  1093117  1096441       0.000113
Fri         4428        95  1171710  1176233       0.000081
Sat         2842        84  1234625  1

### 3.2 전환율 (view→cart, view→purchase)

가설: user-level 은 sparse 라 noisy 가능. item-level smoothing 필요?

In [32]:
# (a) user-level 전환율 — 분모(view) 충분한 user 한정
user_ev = df.groupby(['user_id', 'event_type']).size().unstack(fill_value=0)
user_ev.columns = [f'n_{c}' for c in user_ev.columns]
sub = user_ev[user_ev['n_view'] >= 10].copy()
sub['v2p_rate'] = sub['n_purchase'] / sub['n_view']
sub['v2c_rate'] = sub['n_cart'] / sub['n_view']
print(f'view>=10 user: {len(sub):,}')
n_zero_v2p = (sub['v2p_rate'] == 0).sum()
n_zero_v2c = (sub['v2c_rate'] == 0).sum()
print(f'  v2p_rate=0 인 user: {n_zero_v2p:,} ({100*n_zero_v2p/len(sub):.1f}%)  ← 대부분 0 → user-level v2p 는 noisy')
print(f'  v2c_rate=0 인 user: {n_zero_v2c:,} ({100*n_zero_v2c/len(sub):.1f}%)')

# (b) item-level 전환율 — Bayesian smoothing
item_ev = df.groupby(['item_id', 'event_type']).size().unstack(fill_value=0)
item_ev.columns = [f'n_{c}' for c in item_ev.columns]
global_v2p = item_ev['n_purchase'].sum() / item_ev['n_view'].sum()
alpha = 50  # smoothing strength
item_ev['v2p_smoothed'] = (item_ev['n_purchase'] + alpha * global_v2p) / (item_ev['n_view'] + alpha)

strong = item_ev[item_ev['n_view'] >= 100]
print(f'\nitem-level (view>=100, n={len(strong):,}, α={alpha}, prior={global_v2p:.5f}):')
print(f'  v2p_smoothed 분포: median={strong["v2p_smoothed"].median():.5f}, p90={strong["v2p_smoothed"].quantile(0.9):.5f}, p99={strong["v2p_smoothed"].quantile(0.99):.5f}')
print('\n→ item-level smoothed 은 view>=100 item 에서 안정적, reranker feature 로 활용 가치')


view>=10 user: 238,001
  v2p_rate=0 인 user: 236,843 (99.5%)  ← 대부분 0 → user-level v2p 는 noisy
  v2c_rate=0 인 user: 231,107 (97.1%)

item-level (view>=100, n=12,726, α=50, prior=0.00025):
  v2p_smoothed 분포: median=0.00004, p90=0.00008, p99=0.00365

→ item-level smoothed 은 view>=100 item 에서 안정적, reranker feature 로 활용 가치


## 4. Sequence & Session 패턴 (모델 선택 함의)

### 4.1 cross-session view → purchase

가설: view 와 purchase 가 같은 session 안인가, 다른 session 에서?

In [33]:
sess = df.groupby(['user_id', 'user_session'])['event_time'].agg(start='min', end='max').reset_index()
sess = sess.sort_values(['user_id', 'start'])
sess['next_start'] = sess.groupby('user_id')['start'].shift(-1)
sess['gap_min'] = (sess['next_start'] - sess['end']).dt.total_seconds() / 60
gaps = sess['gap_min'].dropna()
print(f'인접 session gap (분) — n={len(gaps):,}, median={gaps.median():.1f}, mean={gaps.mean():.0f}')

bins   = [-1, 5, 15, 30, 60, 120, 360, 1440, 60*24*7, 10**9]
labels = ['0-5min', '5-15', '15-30', '30-60', '1-2h', '2-6h', '6-24h', '1-7d', '>7d']
print('\ngap bucket 분포:')
print(pd.cut(gaps, bins=bins, labels=labels).value_counts().sort_index())

# view → purchase 전환 패턴
first_view  = (df[df['event_type'] == 'view']
               .groupby(['user_id', 'item_id'])['user_session'].first().rename('view_sess'))
first_purch = (df[df['event_type'] == 'purchase']
               .groupby(['user_id', 'item_id'])['user_session'].first().rename('purch_sess'))
j = pd.concat([first_view, first_purch], axis=1).dropna()
same = (j['view_sess'] == j['purch_sess']).sum()
print(f'\nview→purchase 발생 (user,item) 쌍: {len(j):,}')
print(f'  intra-session: {same} ({100*same/len(j):.1f}%)')
print(f'  inter-session: {len(j)-same} ({100*(len(j)-same)/len(j):.1f}%)  ← session-aware 모델 우선순위 ↓')


인접 session gap (분) — n=2,251,295, median=1279.3, mean=6291

gap bucket 분포:
gap_min
0-5min    355225
5-15      129510
15-30      76280
30-60      69126
1-2h       72054
2-6h      148534
6-24h     320770
1-7d      656702
>7d       413959
Name: count, dtype: int64

view→purchase 발생 (user,item) 쌍: 1,198
  intra-session: 65 (5.4%)
  inter-session: 1133 (94.6%)  ← session-aware 모델 우선순위 ↓
